## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### Operations on compound (multi-dimensional) sets
#### AMPL Formulation

In [11]:
%%ampl_eval
reset;      # clear previous model/data

set ORIG;   # origin nodes
set DEST;   # destination nodes
set PROD;   # product types

# Sparse arc sets (allowed connections)
set LINKS_A within ORIG cross DEST;  # first network
set LINKS_B within ORIG cross DEST;  # second network

# Union and intersection of arc sets
set ALL_LINKS = LINKS_A union LINKS_B;  # all active arcs
set SHARED    = LINKS_A inter LINKS_B;  # common arcs in both networks

# 3D routing structure (origin, destination, product)
set ROUTES = ORIG cross DEST cross PROD;

# Cost and decision variables over 3D routes
param cost {ROUTES} >= 0;
var Ship  {ROUTES} >= 0;

minimize TotalCost:         # Objective: minimize total shipping cost over all routes
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Ship[i,j,p];

In [12]:
%%ampl_eval
data;                   # Start data section

# origin, destination, product sets
set ORIG := O1 O2;
set DEST := D1 D2;
set PROD := P1 P2;

# Sparse network A (allowed arcs)
set LINKS_A := (O1, D1), (O1, D2), (O2, D2);

# Sparse network B (alternative arcs)
set LINKS_B := (O1, D1), (O2, D1), (O2, D2);

# 3D route costs (origin, destination, product)
param cost :=
O1 D1 P1 2
O1 D1 P2 3
O1 D2 P1 4
O1 D2 P2 5
O2 D1 P1 6
O2 D1 P2 7
O2 D2 P1 1
O2 D2 P2 2;

display ALL_LINKS, SHARED, ROUTES;

set ALL_LINKS := (O1,D1) (O1,D2) (O2,D2) (O2,D1);

set SHARED := (O1,D1) (O2,D2);

set ROUTES :=
(O1,D1,P1)   (O1,D2,P1)   (O2,D1,P1)   (O2,D2,P1)
(O1,D1,P2)   (O1,D2,P2)   (O2,D1,P2)   (O2,D2,P2);



#### amplpy Formulation

In [16]:
# Sets
ampl.set['ORIG'] = ['O1', 'O2']
ampl.set['DEST'] = ['D1', 'D2']
ampl.set['PROD'] = ['P1', 'P2']

# Sparse network A
ampl.set['LINKS_A'] = [
    ('O1', 'D1'),
    ('O1', 'D2'),
    ('O2', 'D2')]

# Sparse network B
ampl.set['LINKS_B'] = [
    ('O1', 'D1'),
    ('O2', 'D1'),
    ('O2', 'D2')]

import pandas as pd

# 3D cost parameter (ROUTES = ORIG x DEST x PROD)
ccost = {
    ('O1','D1','P1'): 2,
    ('O1','D1','P2'): 3,
    ('O1','D2','P1'): 4,
    ('O1','D2','P2'): 5,
    ('O2','D1','P1'): 6,
    ('O2','D1','P2'): 7,
    ('O2','D2','P1'): 1,
    ('O2','D2','P2'): 2
}

ampl.param['cost'].set_values(ccost)

# Convert sets to Python tuples
A = set(tuple(r) for r in ampl.set['LINKS_A'].get_values())
B = set(tuple(r) for r in ampl.set['LINKS_B'].get_values())

# ALL_LINKS = union
ALL_LINKS = A | B

# SHARED = intersection
SHARED = A & B

# ROUTES = full 3D product
ROUTES = {
    (i, j, p)
    for i in ampl.set['ORIG'].get_values()
    for j in ampl.set['DEST'].get_values()
    for p in ampl.set['PROD'].get_values()
}

print("ALL_LINKS =", ALL_LINKS)
print("SHARED =", SHARED)
print("ROUTES size =", len(ROUTES))

ALL_LINKS = {('O2', 'D2'), ('O2', 'D1'), ('O1', 'D2'), ('O1', 'D1')}
SHARED = {('O1', 'D1'), ('O2', 'D2')}
ROUTES size = 8
